In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import evaluate
from features import FEATURES, GROUP, TARGET, load_rated, make_splits
from models import fit_predict_type, make_logreg

rated = load_rated()
train, valid, test = make_splits(rated)
dev = pd.concat([train, valid]).reset_index(drop=True)
print(f"Development (train + valid): {len(dev):,}   Test: {len(test):,}")

Development (train + valid): 57,112   Test: 14,278


In [2]:
# Train the chosen model on ALL development data
final_model = make_logreg().fit(dev[FEATURES], dev[TARGET])

# Score the test set: the first and only time it is used
final_score = final_model.predict_proba(test[FEATURES])[:, 1]

rng = np.random.default_rng(0)
test_results = pd.DataFrame([
    evaluate("Random order", test, rng.random(len(test))),
    evaluate("Business type only", test, fit_predict_type(dev, test)),
    evaluate("Logistic regression (final)", test, final_score),
])
print("FINAL RESULTS ON THE LOCKED TEST SET\n")
print(test_results.round(3).to_string(index=False))

FINAL RESULTS ON THE LOCKED TEST SET

                      model  recall@20%  recall@10%  PR-AUC (within)  PR-AUC (pooled)  ROC-AUC (pooled)
               Random order       0.190       0.082            0.062            0.054             0.486
         Business type only       0.326       0.165            0.078            0.079             0.635
Logistic regression (final)       0.393       0.231            0.127            0.157             0.752


In [3]:
# The same result in plain numbers: inspect the top 20% of each borough's list
scored = test.assign(score=final_score)
inspected = found = 0
for _, borough in scored.groupby(GROUP):
    k = int(np.ceil(0.2 * len(borough)))
    inspected += k
    found += borough.nlargest(k, "score")[TARGET].sum()

total_fails = int(test[TARGET].sum())
print(f"Test set: {len(test):,} businesses, {total_fails:,} of them failing.")
print(f"Inspecting the top 20% of each borough's list = {inspected:,} inspections")
print(f"   finds {found:,} of the {total_fails:,} failing businesses ({found / total_fails:.1%}).")
print(f"   Random inspections of the same number would find about {0.2 * total_fails:,.0f}.")

Test set: 14,278 businesses, 801 of them failing.
Inspecting the top 20% of each borough's list = 2,868 inspections
   finds 315 of the 801 failing businesses (39.3%).
   Random inspections of the same number would find about 160.


In [4]:
model_path = PROJECT_ROOT / "models" / "logreg_final.joblib"
joblib.dump(final_model, model_path)
print("Saved to", model_path)

# Check it loads back and gives identical scores
reloaded = joblib.load(model_path)
same = np.allclose(reloaded.predict_proba(test[FEATURES])[:, 1], final_score)
print("Reloaded model gives identical scores:", same)

Saved to c:\projects\food-hygiene-risk-london\models\logreg_final.joblib
Reloaded model gives identical scores: True


## Final result (locked test set, used once)

| Model | recall@20% | recall@10% | PR-AUC (within) |
|---|---|---|---|
| Random order | 0.190 | 0.082 | 0.062 |
| Business type only | 0.326 | 0.165 | 0.078 |
| **Logistic regression (final)** | **0.393** | **0.231** | **0.127** |

- Consistent with cross-validation (recall@20% 0.407-0.415, recall@10% 0.235,
  PR-AUC within 0.130-0.133). The small drop is within normal variation: no overfitting.
- Beats the business-type baseline by +0.067 (recall@20%) and +0.066 (recall@10%).
- **In counts:** 14,278 test businesses, 801 failing. Inspecting the top 20% of each
  borough's list (2,868 inspections) finds **315** failing businesses vs about **160**
  by random selection.
- **Hit rate doubles:** about 1 in 9 model-guided inspections finds a failing business,
  vs about 1 in 18 at random.
- Final model saved to `models/logreg_final.joblib` (reload check: identical scores).